# Week 2: Frame My Lane as an ML Task

**Lane:** CTR / Engagement Opportunity Scoring
**Decision supported:** Which visible pages are under-capturing clicks and should receive metadata or content review?

This notebook frames the task before modeling. The unit is one visible page for one month. An opportunity score helps the SEO team prioritize a finite review queue; it is not an automatic publishing decision.

## 1) My lane as an ML task

**Task type: scoring.** For each visible page and month, assign an opportunity score for how strongly the page appears to under-capture clicks relative to its search visibility.

The score creates an ordered review queue. The team can investigate the highest-scoring pages first, with metadata or content changes as the follow-up action.

## 2) Target or proxy

The eventual target is **future improvement in organic clicks or CTR for the same page during the next observation window**, for example: `future_clicks - current_clicks` or `future_ctr - current_ctr`. This outcome is unavailable at decision time and must be created only after the feature window closes.

For this framing notebook, the target is a **CTR opportunity proxy**: a visible page with enough impressions, a plausible ranking position, and CTR below the expected CTR for that position. The proxy defines the task, but it must be validated against a future-window outcome before model training.

## 3) Success metric

**Primary metric: Precision@10 on the future CTR-improvement label.** Of the ten pages sent to metadata or content review, how many improve clicks or CTR in the next window?

This metric matches the action: the team reviews a short queue. I would also report `NDCG@10` because it rewards putting the strongest opportunities near the top, and compare against a fixed baseline such as impressions multiplied by CTR gap.

In [ ]:
import pandas as pd

# Decision-time fields only. This compact frame uses the intended page-month grain.
lane_df = pd.DataFrame({
    'month': ['2026-03'] * 6,
    'page': ['/flights/manila', '/flights/cebu', '/guides/iloilo-airport',
             '/flights/philippines', '/guides/manila-hotels', '/guides/davao'],
    'impressions': [8200, 5100, 2400, 12800, 3900, 1800],
    'clicks': [246, 204, 120, 256, 117, 72],
    'ctr': [0.030, 0.040, 0.050, 0.020, 0.030, 0.040],
    'avg_position': [8.4, 6.2, 10.1, 12.8, 7.7, 9.5],
})

print(f'Rows: {len(lane_df)}')
print(f'Columns: {lane_df.columns.tolist()}')
lane_df

## 4) The unit of analysis, as a real dataframe

**One row = one visible page observed in one month.** The page identifies the landing page, while impressions, clicks, CTR, and average position describe how that page performed in organic search during the decision window.

The production version should load this same page-month grain from the FlyRank warehouse slice. This local frame makes the grain inspectable even when gated warehouse credentials are not available in the notebook environment. It is intentionally not used to claim model performance.

In [ ]:
key_cols = ['month', 'page']
assert not lane_df.duplicated(key_cols).any(), 'The intended page-month grain must be unique.'

# A framing-only proxy: enough demand, plausible position, and CTR below
# an expected position-adjusted CTR. Training must use future outcomes.
expected_ctr = 0.10 / lane_df['avg_position'].clip(lower=1).pow(0.35)
lane_df['ctr_opportunity_proxy'] = (
    (lane_df['impressions'] >= 4000)
    & (lane_df['avg_position'].between(4, 15))
    & (lane_df['ctr'] < expected_ctr * 0.60)
).astype(int)

print(f'Duplicate page-month keys: {lane_df.duplicated(key_cols).sum()}')
print(f'CTR opportunity proxy positives: {lane_df.ctr_opportunity_proxy.sum()}')
lane_df[key_cols + ['ctr_opportunity_proxy']]

## 5) Why ML beats a fixed rule here

A fixed rule can flag every page below a CTR threshold, but engagement opportunity depends on interacting signals: impression volume, average position, CTR relative to position, page type, query mix, seasonality, and recent trend. The same CTR can mean something different for a page ranking near position 2 versus position 12.

ML is justified if it learns a more useful ordering from historical future outcomes and beats the rule baseline on held-out months. The action remains human review: an SEO practitioner checks the page, title, description, search intent, and content before making a change. I will exclude future clicks, future impressions, and any label-derived field from decision-time features to prevent leakage.

## 6) Self-check

- Task type named: **scoring**, producing an ordered opportunity queue.
- Target/proxy named: **future page-level CTR or click improvement**, with a clearly labeled CTR opportunity proxy.
- Success metric named: **Precision@10**, with NDCG@10 as a supporting metric.
- Unit shown: one **visible page-month** row in a dataframe.
- Action connected: metadata or content review for the highest-scoring pages.
- Leakage boundary stated: future-window and label-derived fields are unavailable at decision time.

**Next step:** replace the compact frame with the authorized FlyRank warehouse slice, build future-window labels by time split, and test whether a learned scorer beats the fixed CTR-gap baseline.